In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# сначала найдём все потенциально невалидные даанные
borders = {"sleep_duration": [0.0, 24.0], "heart_rate": [0.0, 220.0], "bmi": [0.0, 100],
           "calorie_expenditure": [0.0, np.inf], "step_count": [0.0, np.inf], 
           "exercise_duration": [0.0, 1440.0], "water_intake": [0.0, np.inf],
           "diet_type": ("veg", "non-veg", "balanced"), "stress_level": ("low", "high", "medium"),
           "sleep_quality": ('average', 'poor', 'good'), "physical_activity_level": ('sedentary', 'moderate', 'active'),
           "smoking_alcohol": ('yes', 'occasional', 'no'), "gender": ('female', 'other', 'male')}

for col_name in df.columns:
    if col_name not in borders:
        continue
    else:
        try:
            if type(borders[col_name]) is tuple:
                mask = ~(df[col_name].isin(borders[col_name]) | df[col_name].isna())
            elif type(borders[col_name]) is list:
                mask = ~(((df[col_name] >= borders[col_name][0]) & (df[col_name] <= borders[col_name][1])) | df[col_name].isna())
            else:
                raise AttributeError("Нестандартный тип данных из словаря borders")
        except AttributeError as ae:
            print(f"Словарь borders повреждён: {ae}")
        print(f"Столбец {col_name}: {mask.sum()}")
        
            

Столбец sleep_duration: 0
Столбец heart_rate: 0
Столбец bmi: 0
Столбец calorie_expenditure: 0
Столбец step_count: 0
Столбец exercise_duration: 0
Столбец water_intake: 0
Столбец diet_type: 0
Столбец stress_level: 0
Столбец sleep_quality: 0
Столбец physical_activity_level: 0
Столбец smoking_alcohol: 0
Столбец gender: 0


### Вывод
Данные лежат в валидных интервалах

In [4]:
df.isnull().sum()

id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [5]:
missing_per_row = df.isnull().sum(axis=1)
missing_counts = [[num, np.round(num/M, 2)] for num in missing_per_row.value_counts().sort_index()]
print("Количество строк с определенным числом пропусков (кол-во, процент от всего df):")
print(*missing_counts, sep="\n")

Количество строк с определенным числом пропусков (кол-во, процент от всего df):
[349623, np.float64(0.51)]
[248134, np.float64(0.36)]
[77311, np.float64(0.11)]
[13445, np.float64(0.02)]
[1479, np.float64(0.0)]
[87, np.float64(0.0)]
[9, np.float64(0.0)]


In [6]:
# так как общий процент сэмплов с 3+ пропусками <= 0.2, то откинем их
df = df[~(df.isnull().sum(axis=1) >= 3)]

Дальше надо решить проблему обработки категориальных признаков. Воспользуемся One Hot Encoding

In [7]:
# Заменяем NaN на медианные значения
from sklearn.preprocessing import OneHotEncoder

columns_categor = ["diet_type", "stress_level", "sleep_quality", "physical_activity_level", "smoking_alcohol", "gender"]

df_Y = df[["id", "health_condition"]].copy()
df.drop(columns=["id", "health_condition"], inplace=True)

df_numerical = df.drop(columns=columns_categor)
df_numerical = df_numerical.fillna(df_numerical.median())

df_categorical = df[columns_categor].fillna("Missing")
encoder = OneHotEncoder(sparse_output=False)
df_categorical_encoded = encoder.fit_transform(df_categorical)

df_categorical = pd.DataFrame(
    df_categorical_encoded,
    columns=encoder.get_feature_names_out(columns_categor),
    index=df_categorical.index 
)

df = pd.concat([df_Y, df_numerical, df_categorical], axis=1)

print(df.isna().sum())

id                                   0
health_condition                     0
sleep_duration                       0
heart_rate                           0
bmi                                  0
calorie_expenditure                  0
step_count                           0
exercise_duration                    0
water_intake                         0
diet_type_Missing                    0
diet_type_balanced                   0
diet_type_non-veg                    0
diet_type_veg                        0
stress_level_Missing                 0
stress_level_high                    0
stress_level_low                     0
stress_level_medium                  0
sleep_quality_Missing                0
sleep_quality_average                0
sleep_quality_good                   0
sleep_quality_poor                   0
physical_activity_level_Missing      0
physical_activity_level_active       0
physical_activity_level_moderate     0
physical_activity_level_sedentary    0
smoking_alcohol_Missing  

#### Разделим данные

In [8]:
df = df.drop(["id"], axis=1)
df['health_condition'] = df['health_condition'].map({'at-risk': 0, 'unhealthy': 1, 'fit': 2})

#### На 3 выборки: train(70%), validate(20%), test(10%)

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['health_condition'])
y = df['health_condition']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.3,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.3333,
    random_state=42,
    stratify=y_temp
)


df_train = pd.concat([X_train, y_train], axis=1)
df_val = pd.concat([X_val, y_val], axis=1)
df_test = pd.concat([X_test, y_test], axis=1)